# Use and adapt the model

This notebook shows, in six short steps, how to use this model for your own question: from reproducing the results to replacing an ageing law. The explanations are in [`docs/ADAPT.md`](../docs/ADAPT.md).

Run the cells in order (Shift + Enter). The notebook works from the `examples` folder of the repository.

## Setup

In [ ]:
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"           # the h2ageing package of this repository
sys.path.insert(0, str(SRC))

import numpy as np
from h2ageing import load_config, with_values, run

cfg = load_config()                         # every input, from config/default.yaml

## Level 1: reproduce the results

In [ ]:
ref = run(cfg)
ref["chain"][["case", "total_g_per_km", "rise_pct"]].round(1)

## Level 2: ask the model a question

*What if the fuel cell stack lasts 5 000 h?* `with_values` returns a modified copy of the configuration.

In [ ]:
r = run(with_values(cfg, **{"cases.central.fuel_cell.hours_to_end_of_life": 5000.0}))
r["chain"][["case", "total_g_per_km", "rise_pct"]].round(1)

*From which lifetime is one stack enough?*

In [ ]:
for d in [3000, 4100, 5000, 5554, 6000, 7000]:
    x = run(with_values(cfg, **{"cases.central.fuel_cell.hours_to_end_of_life": float(d)}))
    stacks = x["life"].set_index("case").loc["central", "fuel_cell_stacks"]
    rise = x["chain"].set_index("case").loc["central", "rise_pct"]
    print(f"{d} h -> {stacks:.0f} stack(s), chain rise +{rise:.1f} %")

Below 5 554 h (the operating hours of the vehicle), two stacks are needed and the result barely moves; above it, one stack is enough.

## Level 3: your own case

A manufacturer whose stack lasts 7 000 h and whose electrolyser runs on wind electricity. Several values can be changed at once; the same can be written in a copy of `config/default.yaml` and run with `python scripts/run.py --config config/my_case.yaml`.

In [ ]:
my_case = with_values(cfg, **{
    "cases.central.fuel_cell.hours_to_end_of_life": 7000.0,   # own endurance test
    "usage.electricity_electrolyser": "wind_FR",              # wind power purchase agreement
})
run(my_case)["chain"][["case", "total_g_per_km", "rise_pct"]].round(1)

## Level 4: your own ageing constants

A voltage loss of 27 mV at 1.8 A/cm² after 1 000 h, measured on another cell, instead of 45 mV in the pessimistic case.

In [ ]:
r = run(with_values(cfg, **{"cases.pessimistic.fuel_cell.voltage_loss_at_1p8_A_cm2_1000h_V": 0.027}))
r["life"][["case", "fuel_cell_end_of_life_h", "fuel_cell_stacks"]].round(0)

## Level 5: another ageing law

A fuel cell resistance growing as the square root of time, with the same value as the linear law at 1 000 h. Only the law is written; end of life, stacks and LCA are recomputed.

In [ ]:
from h2ageing import pemfc

original_law = pemfc.FuelCell.resistance

def my_resistance(self, t, p):
    t = np.asarray(t, float)
    return self.R0 * (1 + p.get("k_R", 0.0) * 1000 * np.sqrt(t / 1000))

pemfc.FuelCell.resistance = my_resistance
r = run(cfg)
pemfc.FuelCell.resistance = original_law        # put the original law back

r["life"][["case", "fuel_cell_end_of_life_h", "fuel_cell_stacks"]].round(0)

In [ ]:
r["chain"][["case", "total_g_per_km", "rise_pct"]].round(1)

## Level 6: your own data

Measurements, inventories and impacts are plain files in `data/`. Replace them, refit or rebuild as explained in [`docs/ADAPT.md`](../docs/ADAPT.md) (level 6) and [`docs/CUSTOMISE.md`](../docs/CUSTOMISE.md).

## Check

The reference results are unchanged as long as the default configuration and the original laws are used:

In [ ]:
assert abs(run(cfg)["chain"].set_index("case").loc["central", "rise_pct"] - 31.07) < 0.01
print("reference results unchanged")